# 02 — Data Preprocessing & Feature Engineering

**职责**: 数据清洗 → 缺失值处理 → 特征构建 → 特征选择 → 输出处理后的特征矩阵

**输入**: `data/raw/train_data.csv`, `data/raw/test_data.csv`  
**输出**: `data/processed/train_processed.parquet`

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np

from config import DATA_RAW, DATA_TEST, DATA_PROC, TARGET
from src.preprocessing import load_and_clean, fill_missing, build_feature_matrix
from src.feature_engineering import select_features, compute_ensemble_importance

## 1. 加载原始数据 & 基础清洗

In [2]:
df = load_and_clean(DATA_RAW)
print(f"Shape: {df.shape}")
print(f"Missing values: {df.isnull().sum().sum()}")
df.head()

Shape: (375734, 251)
Missing values: 0


,index,lat,lon,startdate,contest-pevpr-sfc-gauss-14d__pevpr,nmme0-tmp2m-34w__cancm30,nmme0-tmp2m-34w__cancm40,nmme0-tmp2m-34w__ccsm30,nmme0-tmp2m-34w__ccsm40,nmme0-tmp2m-34w__cfsv20,...,wind-vwnd-925-2010-16,wind-vwnd-925-2010-17,wind-vwnd-925-2010-18,wind-vwnd-925-2010-19,wind-vwnd-925-2010-20,year,month,day,week,season
0,0,0.0,0.833333,2014-09-01,237.00,29.02,31.64,29.57,30.73,29.71,...,48.13,28.09,-13.50,11.90,4.58,2014,9,1,36,Fall
1,1,0.0,0.833333,2014-09-02,228.90,29.02,31.64,29.57,30.73,29.71,...,48.60,27.41,-23.77,15.44,3.42,2014,9,2,36,Fall
2,2,0.0,0.833333,2014-09-03,220.69,29.02,31.64,29.57,30.73,29.71,...,48.53,19.21,-33.16,15.11,4.82,2014,9,3,36,Fall
3,3,0.0,0.833333,2014-09-04,225.28,29.02,31.64,29.57,30.73,29.71,...,50.59,8.29,-37.22,18.24,9.74,2014,9,4,36,Fall
4,4,0.0,0.833333,2014-09-05,237.24,29.02,31.64,29.57,30.73,29.71,...,54.73,-2.58,-42.30,21.91,10.95,2014,9,5,36,Fall


## 2. 缺失值处理

检查填充后的数据质量。如果需要更精细的策略，可以修改 `src/preprocessing.py` 中的 `fill_missing()`。

In [3]:
missing_after = df.isnull().sum()
print(f"Remaining missing columns: {(missing_after > 0).sum()}")
if (missing_after > 0).any():
    print(missing_after[missing_after > 0].sort_values(ascending=False).head(10))

Remaining missing columns: 0


## 3. PCA

分组pca

In [4]:
"""
WiDS Datathon 2023 — 数据预处理 & 特征工程
"""

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────
# 配置
# ─────────────────────────────────────────────────────────────
RAW_DIR  = DATA_RAW.parent
SAVE_DIR = DATA_PROC.parent

TARGET      = "contest-tmp2m-14d__tmp2m"
TRAIN_RATIO = 0.8
PCA_VAR     = 0.95
RANDOM_SEED = 42


# ─────────────────────────────────────────────────────────────
# 1. 加载原始数据
# ─────────────────────────────────────────────────────────────
def load_raw(path: Path) -> pd.DataFrame:
    print(f"📂 加载数据: {path.name}")
    df = pd.read_csv(path)
    df["startdate"] = pd.to_datetime(df["startdate"])
    df = df.sort_values("startdate").reset_index(drop=True)
    print(f"   shape: {df.shape}")

    if TARGET not in df.columns:
        candidates = [c for c in df.columns if "tmp2m" in c]
        print(f"\n⚠️  未找到 TARGET='{TARGET}'，数据中含 'tmp2m' 的列：")
        for c in candidates:
            print(f"   {c}")
        raise KeyError(f"TARGET 列 '{TARGET}' 不存在，请确认列名后修改顶部 TARGET 变量")

    return df


# ─────────────────────────────────────────────────────────────
# 2. 特征分类
# ─────────────────────────────────────────────────────────────
def categorize_features(df: pd.DataFrame):
    all_cols = df.columns.tolist()

    # 直接保留的强信号特征（EDA 证实与 target 强相关）
    direct_keep_prefixes = [
        "contest-",
        "nmme-tmp2m-34w__nmmemean",
        "nmme-tmp2m-56w__nmmemean",
    ]
    direct_keep = [
        c for c in all_cols
        if any(c.startswith(p) for p in direct_keep_prefixes)
        and c != TARGET
    ]

    # PCA 压缩的高维冗余特征组
    pca_groups = {
        "nmme_tmp34w": [c for c in all_cols
                        if c.startswith(("nmme-tmp2m-34w__", "nmme0-tmp2m-34w__"))
                        and "nmmemean" not in c],
        "nmme_tmp56w": [c for c in all_cols
                        if c.startswith("nmme-tmp2m-56w__")
                        and "nmmemean" not in c],
        "nmme_prate":  [c for c in all_cols
                        if any(c.startswith(p) for p in [
                            "nmme-prate-34w__", "nmme0-prate-34w__",
                            "nmme-prate-56w__", "nmme0-prate-56w__"])],
        "hgt_10":  [c for c in all_cols if c.startswith("wind-hgt-10-2010-")],
        "hgt_100": [c for c in all_cols if c.startswith("wind-hgt-100-2010-")],
        "hgt_500": [c for c in all_cols if c.startswith("wind-hgt-500-2010-")],
        "hgt_850": [c for c in all_cols if c.startswith("wind-hgt-850-2010-")],
        "sst":     [c for c in all_cols if c.startswith("sst-2010-")],
        "icec":    [c for c in all_cols if c.startswith("icec-2010-")],
        "wind":    [c for c in all_cols if any(c.startswith(p) for p in [
                    "wind-uwnd-250-2010-", "wind-vwnd-250-2010-",
                    "wind-uwnd-925-2010-", "wind-vwnd-925-2010-"])],
        "tele":    [c for c in all_cols if c.startswith(("mei__", "mjo1d__"))],
    }
    pca_groups = {k: [c for c in v if c in all_cols] for k, v in pca_groups.items()}
    pca_groups = {k: v for k, v in pca_groups.items() if len(v) >= 2}

    static_cols = [c for c in ["lat", "lon", "elevation__elevation",
                                "climateregions__climateregion"] if c in all_cols]

    ffill_cols = [c for c in all_cols
                  if c.startswith(("nmme-tmp2m-34w__nmmemean",
                                   "nmme-tmp2m-56w__nmmemean"))]

    return direct_keep, pca_groups, static_cols, ffill_cols


# ─────────────────────────────────────────────────────────────
# 3. 缺失值填补（先切分再 fit，无泄露）
# ─────────────────────────────────────────────────────────────
def impute(train_df: pd.DataFrame, test_df: pd.DataFrame,
           ffill_cols: list, median_cols: list):
    print("\n🔧 缺失值填补...")

    valid_ffill = [c for c in ffill_cols if c in train_df.columns]
    if valid_ffill:
        train_df[valid_ffill] = train_df[valid_ffill].ffill()
        test_df[valid_ffill]  = test_df[valid_ffill].ffill()
        for col in valid_ffill:
            med = train_df[col].median()
            train_df[col] = train_df[col].fillna(med)
            test_df[col]  = test_df[col].fillna(med)

    valid_median = [c for c in median_cols
                    if c in train_df.columns and train_df[c].dtype != object]
    if valid_median:
        imp = SimpleImputer(strategy="median")
        train_df[valid_median] = imp.fit_transform(train_df[valid_median])
        test_df[valid_median]  = imp.transform(test_df[valid_median])

    print(f"   填补后 train NaN: {train_df.isnull().sum().sum()}"
          f" | test NaN: {test_df.isnull().sum().sum()}")
    return train_df, test_df


# ─────────────────────────────────────────────────────────────
# 4. 分组 PCA（动态组件数，fit on train only）
# ─────────────────────────────────────────────────────────────
def apply_grouped_pca(train_df: pd.DataFrame, test_df: pd.DataFrame,
                      pca_groups: dict, var_threshold: float = PCA_VAR):
    print(f"\n📐 分组 PCA（保留 {var_threshold*100:.0f}% 方差）...")

    train_pca, test_pca = {}, {}

    for name, cols in pca_groups.items():
        valid = [c for c in cols if c in train_df.columns]
        if len(valid) < 2:
            continue

        X_tr = train_df[valid].values
        X_te = test_df[valid].values

        cumvar = np.cumsum(
            PCA(random_state=RANDOM_SEED).fit(X_tr).explained_variance_ratio_
        )
        n_comp = min(max(1, int(np.searchsorted(cumvar, var_threshold) + 1)),
                     len(valid))

        pca    = PCA(n_components=n_comp, random_state=RANDOM_SEED)
        tr_out = pca.fit_transform(X_tr)
        te_out = pca.transform(X_te)

        for i in range(n_comp):
            col = f"pca_{name}_{i+1}"
            train_pca[col] = tr_out[:, i]
            test_pca[col]  = te_out[:, i]

        print(f"  {name:15s}: {len(valid):3d}列 → {n_comp}个主成分"
              f"（解释方差 {pca.explained_variance_ratio_.sum()*100:.1f}%）")

    return (pd.DataFrame(train_pca, index=train_df.index),
            pd.DataFrame(test_pca,  index=test_df.index))


# ─────────────────────────────────────────────────────────────
# 5. 时间特征工程
# ─────────────────────────────────────────────────────────────
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    d    = df["startdate"]
    df   = df.copy()
    week = d.dt.isocalendar().week.astype(int)
    df["year"]       = d.dt.year
    df["month"]      = d.dt.month
    df["day"]        = d.dt.day
    df["dayofyear"]  = d.dt.dayofyear
    df["weekofyear"] = week
    df["sin_doy"]    = np.sin(2 * np.pi * d.dt.dayofyear / 365)
    df["cos_doy"]    = np.cos(2 * np.pi * d.dt.dayofyear / 365)
    df["sin_month"]  = np.sin(2 * np.pi * d.dt.month / 12)
    df["cos_month"]  = np.cos(2 * np.pi * d.dt.month / 12)
    df["sin_week"]   = np.sin(2 * np.pi * week / 52)
    df["cos_week"]   = np.cos(2 * np.pi * week / 52)
    return df


# ─────────────────────────────────────────────────────────────
# 6. 主流程
# ─────────────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  WiDS 2023 — 数据预处理 & 特征工程")
    print("=" * 60)

    df = load_raw(RAW_DIR / "train_data.csv")

    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print(f"\n📊 缺失值: {len(missing)} 列有缺失，共 {missing.sum()} 个")
    if len(missing):
        print(missing.sort_values(ascending=False).head(10).to_string())

    direct_keep, pca_groups, static_cols, ffill_cols = categorize_features(df)
    print(f"\n📋 特征分类:")
    print(f"   直接保留: {len(direct_keep)} 列")
    print(f"   PCA 压缩: {len(pca_groups)} 组"
          f"（{sum(len(v) for v in pca_groups.values())} 列）")
    print(f"   静态地理: {len(static_cols)} 列")

    # 先按时间切分，再 fit（避免数据泄露）
    split_idx = int(len(df) * TRAIN_RATIO)
    train_raw = df.iloc[:split_idx].copy()
    test_raw  = df.iloc[split_idx:].copy()
    print(f"\n✂️  时间切分:")
    print(f"   train: {train_raw['startdate'].min().date()} ~ "
          f"{train_raw['startdate'].max().date()}  ({len(train_raw)} 行)")
    print(f"   test : {test_raw['startdate'].min().date()} ~ "
          f"{test_raw['startdate'].max().date()}  ({len(test_raw)} 行)")

    all_pca_cols = [c for cols in pca_groups.values() for c in cols]
    median_cols  = list(set(all_pca_cols + direct_keep) - set(ffill_cols))
    train_raw, test_raw = impute(train_raw, test_raw, ffill_cols, median_cols)

    train_pca_df, test_pca_df = apply_grouped_pca(train_raw, test_raw, pca_groups)

    train_raw = add_time_features(train_raw)
    test_raw  = add_time_features(test_raw)

    time_cols = ["year", "month", "day", "dayofyear", "weekofyear",
                 "sin_doy", "cos_doy", "sin_month", "cos_month",
                 "sin_week", "cos_week"]
    keep_cols = list(dict.fromkeys(
        [c for c in static_cols + direct_keep + time_cols
         if c in train_raw.columns]
    ))

    train_final = pd.concat([
        train_raw[keep_cols].reset_index(drop=True),
        train_pca_df.reset_index(drop=True),
    ], axis=1)
    test_final = pd.concat([
        test_raw[keep_cols].reset_index(drop=True),
        test_pca_df.reset_index(drop=True),
    ], axis=1)

    # startdate 保留到 processed 文件，供 feature_engineering.py 使用
    train_final["startdate"] = train_raw["startdate"].to_numpy()
    test_final["startdate"]  = test_raw["startdate"].to_numpy()

    train_final[TARGET] = train_raw[TARGET].to_numpy()
    test_final[TARGET]  = test_raw[TARGET].to_numpy()

    print(f"\n✅ 最终特征数: train {train_final.shape} | test {test_final.shape}")

    train_path = SAVE_DIR / "train.csv"
    test_path  = SAVE_DIR / "test.csv"
    train_final.to_csv(train_path, index=False)
    test_final.to_csv(test_path,   index=False)
    print(f"\n💾 已保存: {train_path}")
    print(f"           {test_path}")
    print("\n🎉 预处理完成！接下来运行 feature_engineering.py")


if __name__ == "__main__":
    main()

  WiDS 2023 — 数据预处理 & 特征工程
📂 加载数据: train_data.csv
   shape: (375734, 246)

📊 缺失值: 8 列有缺失，共 101772 个
nmme0-tmp2m-34w__ccsm30    15934
nmme0-prate-56w__ccsm30    15934
nmme0-prate-34w__ccsm30    15934
ccsm30                     15934
nmme-tmp2m-56w__ccsm3      10280
nmme-prate-56w__ccsm3      10280
nmme-prate-34w__ccsm3       8738
nmme-tmp2m-34w__ccsm3       8738

📋 特征分类:
   直接保留: 16 列
   PCA 压缩: 11 组（213 列）
   静态地理: 4 列

✂️  时间切分:
   train: 2014-09-01 ~ 2016-04-07  (300587 行)
   test : 2016-04-07 ~ 2016-08-31  (75147 行)

🔧 缺失值填补...
   填补后 train NaN: 15934 | test NaN: 0

📐 分组 PCA（保留 95% 方差）...
  nmme_tmp34w    :  19列 → 2个主成分（解释方差 96.8%）
  nmme_tmp56w    :   9列 → 1个主成分（解释方差 96.9%）
  nmme_prate     :  40列 → 12个主成分（解释方差 95.3%）
  hgt_10         :  10列 → 2个主成分（解释方差 97.0%）
  hgt_100        :  10列 → 2个主成分（解释方差 95.8%）
  hgt_500        :  10列 → 6个主成分（解释方差 96.0%）
  hgt_850        :  10列 → 8个主成分（解释方差 96.9%）
  sst            :  10列 → 2个主成分（解释方差 96.3%）
  icec           :  10列 → 3个主成分（解释方差 96.6%）
  wi

## 5. 特征工程

In [5]:
"""
WiDS 2023 — 补充特征工程模块
在 preprocess.py 生成 train.csv / test.csv 之后、模型训练之前调用
直接在已有特征基础上新增强信号特征，覆盖保存原文件

新增特征类型：
  1. 气候区 × 月份 历史气温统计（均值/标准差/中位数）→ 最强信号
  2. 经纬度网格 × 月份 历史气温统计
  3. 滚动窗口统计（按 lat+lon 分组，7/14/30 天均值）
  4. 与历史均值的偏差特征（捕捉异常天气）
  5. 经纬度交叉特征
"""

import pandas as pd
import numpy as np
from pathlib import Path

# ── 路径配置（与 train.py 保持一致）──────────────────────────
BASE_DIR   = Path(r"D:\catboost\weather-forecast-main")
TRAIN_PATH = BASE_DIR / "data" / "processed" / "train.csv"
TEST_PATH  = BASE_DIR / "data" / "processed" / "test.csv"

TARGET = "contest-tmp2m-14d__tmp2m"


def add_features(train: pd.DataFrame, test: pd.DataFrame):
    """
    所有统计特征只在 train 上 fit，再 merge 到 test，避免泄露。
    """
    print("⚙️  补充特征工程...")
    orig_train_cols = train.shape[1]

    # ── 确保时间列存在 ────────────────────────────────────────
    for df in [train, test]:
        if "month" not in df.columns and "startdate" in df.columns:
            df["startdate"] = pd.to_datetime(df["startdate"])
            df["month"]     = df["startdate"].dt.month

    # ════════════════════════════════════════════════════════
    # 1. 气候区 × 月份 历史气温统计（最强特征）
    # ════════════════════════════════════════════════════════
    if "climateregions__climateregion" in train.columns and "month" in train.columns:
        grp = train.groupby(["climateregions__climateregion", "month"])[TARGET].agg(
            climate_month_mean   = "mean",
            climate_month_std    = "std",
            climate_month_median = "median",
            climate_month_q10    = lambda x: x.quantile(0.1),
            climate_month_q90    = lambda x: x.quantile(0.9),
        ).reset_index()

        train = train.merge(grp, on=["climateregions__climateregion", "month"], how="left")
        test  = test.merge(grp,  on=["climateregions__climateregion", "month"], how="left")

        # 与历史均值的偏差（捕捉相对于气候均值的异常）
        # 只能在 train 上算（test 没有真实 target）
        train["climate_month_anomaly"] = train[TARGET] - train["climate_month_mean"]

        print("   ✓ 气候区×月份统计特征")

    # ════════════════════════════════════════════════════════
    # 2. 经纬度网格 × 月份 历史气温统计
    # ════════════════════════════════════════════════════════
    if "lat" in train.columns and "lon" in train.columns:
        # 将经纬度离散化为粗网格（约 2° × 2° 分辨率）
        for df in [train, test]:
            df["lat_grid"] = (df["lat"] // 2 * 2).astype(int)
            df["lon_grid"] = (df["lon"] // 2 * 2).astype(int)

        grp2 = train.groupby(["lat_grid", "lon_grid", "month"])[TARGET].agg(
            grid_month_mean   = "mean",
            grid_month_std    = "std",
            grid_month_median = "median",
        ).reset_index()

        train = train.merge(grp2, on=["lat_grid", "lon_grid", "month"], how="left")
        test  = test.merge(grp2,  on=["lat_grid", "lon_grid", "month"], how="left")

        # 经纬度交叉特征
        for df in [train, test]:
            df["lat_lon_interact"] = df["lat"] * df["lon"]
            df["lat_abs"]          = df["lat"].abs()
            # 距赤道距离（纬度绝对值越大，气温越低）
            df["dist_equator"]     = df["lat"].abs()

        print("   ✓ 经纬度网格×月份统计特征 + 交叉特征")

    # ════════════════════════════════════════════════════════
    # 3. 按 (lat, lon) 分组的滚动窗口统计
    #    注意：只在 train 内部做，test 用 train 的统计填充
    # ════════════════════════════════════════════════════════
    if "lat" in train.columns and "lon" in train.columns:
        # 确保 startdate 存在用于排序
        if "startdate" in train.columns:
            train["startdate"] = pd.to_datetime(train["startdate"])
            train = train.sort_values(["lat", "lon", "startdate"])

            # 按地点分组，计算滚动 14 天和 30 天的目标均值
            # shift(1) 避免当天信息泄露
            for window in [14, 30]:
                col_name = f"rolling_{window}d_mean"
                train[col_name] = (
                    train.groupby(["lat", "lon"])[TARGET]
                    .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
                )

            # test 用 train 中各地点最近的滚动均值填充
            last_rolling = (
                train.groupby(["lat", "lon"])[["rolling_14d_mean", "rolling_30d_mean"]]
                .last()
                .reset_index()
            )
            test = test.merge(last_rolling, on=["lat", "lon"], how="left")

            print("   ✓ 滚动窗口统计特征（14d / 30d）")

    # ════════════════════════════════════════════════════════
    # 4. 季节特征（春/夏/秋/冬）
    # ════════════════════════════════════════════════════════
    def month_to_season(m):
        if m in [12, 1, 2]:  return 0   # 冬
        if m in [3, 4, 5]:   return 1   # 春
        if m in [6, 7, 8]:   return 2   # 夏
        return 3                         # 秋

    if "month" in train.columns:
        for df in [train, test]:
            df["season"] = df["month"].map(month_to_season)
        print("   ✓ 季节特征")

    # ════════════════════════════════════════════════════════
    # 5. 缺失值填充（新增特征可能有 NaN）
    # ════════════════════════════════════════════════════════
    new_feat_cols = [c for c in train.columns if c not in
                     pd.read_csv(TRAIN_PATH, nrows=0).columns]

    for col in new_feat_cols:
        if train[col].isnull().any():
            fill_val = train[col].median()
            train[col] = train[col].fillna(fill_val)
            test[col]  = test[col].fillna(fill_val)

    added = train.shape[1] - orig_train_cols
    print(f"\n   新增特征数量: {added}")
    print(f"   最终特征总数: train {train.shape} | test {test.shape}")

    return train, test


def main():
    print("=" * 55)
    print("  WiDS 2023 — 补充特征工程")
    print("=" * 55)

    print("📂 加载已处理数据...")
    train = pd.read_csv(TRAIN_PATH)
    test  = pd.read_csv(TEST_PATH)
    print(f"   train: {train.shape} | test: {test.shape}")

    train, test = add_features(train, test)

    # 覆盖保存
    train.to_csv(TRAIN_PATH, index=False)
    test.to_csv(TEST_PATH,   index=False)
    print(f"\n💾 已覆盖保存:")
    print(f"   {TRAIN_PATH}")
    print(f"   {TEST_PATH}")
    print("\n🎉 特征工程完成！现在可以运行 train.py")


if __name__ == "__main__":
    main()

  WiDS 2023 — 补充特征工程
📂 加载已处理数据...


FileNotFoundError: [Errno 2] No such file or directory: 'D:\\catboost\\weather-forecast-main/data/processed/train.csv'